In [0]:
%run ./config

In [0]:
dbutils.widgets.text("batch_control_key", "", "Batch Control Key")

BATCH_CONTROL_KEY = dbutils.widgets.get("batch_control_key").strip()
assert BATCH_CONTROL_KEY, "batch_control_key parameter is required"


In [0]:

# ===================================================================== #
# Read active config and extract silver-layer parameters                #
# ===================================================================== #
config_df = (
    spark.read.table(STREAMING_METADATA_TABLE)
    .filter(
        (F.col("batch_control_key") == BATCH_CONTROL_KEY) &
        (F.col("is_active") == True)
    )
)

assert config_df.count() > 0, (
    f"No active config found for '{BATCH_CONTROL_KEY}' in {STREAMING_METADATA_TABLE}"
)

config = config_df.collect()[0].asDict()

bronze_table_name = config["bronze_table_name"]
silver_table_name = config.get("silver_table_name")
silver_custom_notebook_url = (config.get("silver_custom_notebook_url") or "").strip()
partition_column = config.get("partition_column")
data_format = (config.get("data_format") or "JSON").strip().upper()
xml_row_tag = config.get("xml_row_tag", "row") or "row"
xsd_schema_path = (config.get("xsd_schema_path") or "").strip() or None
source_system = config["source_system"]

AUDIT_BATCH_KEY = f"{BATCH_CONTROL_KEY}_SILVER"

print(f" Bronze Table        : {bronze_table_name}")
print(f" Silver Table        : {silver_table_name}")
print(f" Data Format         : {data_format}")
print(f" XSD Schema Path     : {xsd_schema_path or '(none)'}")
print(f" Custom Notebook     : {silver_custom_notebook_url or '(direct write)'}")
print(f" Source System       : {source_system}")

if not silver_table_name:
    print("\n \u26a0 silver_table_name not configured. Nothing to process.")
    dbutils.notebook.exit(json.dumps({"status": "SKIPPED", "message": "No silver_table_name configured"}))
